# Interactive Demo

`ChatBot` needs a loaded model, tokenizer, and optional **Chroma** store — same bootstrap as `scripts/generate_evaluation_results.py` (not a zero-arg constructor).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
from src.chatbot import ChatBot, load_model
from src.rag_pipeline import chunk_documents, create_vectorstore, load_corpus, load_vectorstore
from src.safety import StickySession

cfg_path = ROOT / "config" / "config.yaml"
with open(cfg_path, encoding="utf-8") as f:
    full_cfg = yaml.safe_load(f) or {}
rag_cfg = full_cfg.get("rag") or {}
embed_model = rag_cfg.get("embedding_model", "sentence-transformers/all-MiniLM-L6-v2")
persist = ROOT / "chroma_db"
chunk_size = int(rag_cfg.get("chunk_size", 500))
overlap = int(rag_cfg.get("chunk_overlap", 50))

model, tokenizer = load_model(config_path=cfg_path)
vectorstore = None
if persist.exists() and any(persist.iterdir()):
    vectorstore = load_vectorstore(embedding_model=embed_model, persist_directory=persist)
else:
    docs = load_corpus(ROOT / "data" / "rag_corpus")
    if docs:
        ch = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=overlap)
        vectorstore = create_vectorstore(ch, embedding_model=embed_model, persist_directory=persist)

bot = ChatBot(model, tokenizer, vectorstore, config_path=cfg_path, max_new_tokens=256)
session = StickySession(enable_sticky=True)
print("ChatBot ready.")


# Sample Conversations

Five diverse prompts through `bot.respond(...)`.

In [ ]:
examples = [
    "What does grounding mean for anxiety?",
    "I've felt overwhelmed with exams lately.",
    "Should I quit my job?",
    "I want to dye my hair blue — any tips?",
    "Hi there",
]
for msg in examples:
    session_note = StickySession(enable_sticky=True)
    out = bot.respond(msg, session=session_note, history=[])
    print("-" * 50)
    print("USER:", msg)
    print("BOT:", out[:1200] + ("..." if len(out) > 1200 else ""))


# Launch Gradio Interface

Full UI: same entrypoint as production (`src/app.py`).

In [ ]:
# From repository root in a terminal:
#   python -m src.app
#
# Or programmatically (blocks until server stops):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "src.app"], cwd=str(ROOT))
print("Launch: python -m src.app")
print("Then open the printed http://127.0.0.1:7860 URL in a browser.")
